# 🧪 HARD QUERY TEST — YOLOv8 + Supervision · 3 chế độ đếm

Test đếm trên **tất cả video** với **YOLOv8** (bbox ổn định) kết hợp **supervision**
(ByteTrack + LineZone/PolygonZone). Mỗi video chạy **3 chế độ**:

1. **LINE** — đếm IN/OUT qua vạch (`sv.LineZone`)
2. **ZONE** — đếm trong vùng (`sv.PolygonZone`)
3. **FULL** — đếm toàn màn hình (unique tracks)

Output: **ảnh lưới nhận diện** + **video annotated** cho từng trường hợp.
Cần **GPU** (nhanh hơn) hoặc **CPU** (YOLO chạy được cả hai).

In [ ]:
# Cell 1 — Cài thư viện (fix Pillow cho Kaggle)
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless -q 2>/dev/null
!pip install -q -U "Pillow>=11.1.0" "opencv-python-headless==4.11.0.86"     ultralytics "supervision>=0.21" matplotlib
print("✅ Đã cài. Restart kernel (Run ▸ Restart) rồi chạy tiếp từ Cell 2.")

In [ ]:
# Cell 2 — Imports + kiểm tra GPU
# Auto-install nếu Cell 1 chưa chạy hoặc kernel chưa restart
import subprocess, sys
for pkg in ['ultralytics', 'supervision', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'📦 Đang cài {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
try:
    from PIL._typing import _Ink
except ImportError:
    print('📦 Upgrade Pillow...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'Pillow>=11.1.0'])

import os, time, math, subprocess, urllib.request, warnings
import numpy as np, cv2, torch
import supervision as sv
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Video, display, Markdown, Image as IPImage

warnings.filterwarnings("ignore")
print("supervision:", sv.__version__)
print("torch      :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU        :", p.name, f"{p.total_memory/1024**3:.1f}GB")

In [ ]:
# Cell 3 — Nạp YOLOv8
from ultralytics import YOLO

MODEL_NAME = "yolov8m.pt"
model = YOLO(MODEL_NAME)
CLASS_NAMES = model.names
print(f"✅ Loaded {MODEL_NAME} — {len(CLASS_NAMES)} classes")
print("COCO classes:", list(CLASS_NAMES.values())[:20], "...")

In [ ]:
# Cell 4 — Hàm detect chung: YOLO → sv.Detections (đã lọc class)
CONF_THRESHOLD = 0.25

def yolo_detect(frame, class_names=None, conf=CONF_THRESHOLD):
    results = model(frame, conf=conf, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    if class_names and len(detections) > 0:
        wanted_ids = {k for k, v in CLASS_NAMES.items() if v in class_names}
        mask = np.array([cid in wanted_ids for cid in detections.class_id])
        detections = detections[mask]
    return detections

def get_class_labels(detections):
    if detections.class_id is None:
        return ["?" for _ in range(len(detections))]
    return [CLASS_NAMES.get(int(cid), "?") for cid in detections.class_id]

print("✅ Hàm detect sẵn sàng.")

In [ ]:
# Cell 5 — Cấu hình VIDEO + 3 chế độ đếm
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())

REPO = os.path.join(WORK, "VisionOS")
BR = "claude/locate-anything-test-suite-xwju2f"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BR,
                    "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git", REPO], check=True)
VID_DIR = os.path.join(REPO, "VisionOS", "sample_videos")

def _dl(name):
    p = os.path.join(WORK, name)
    if not os.path.exists(p):
        try:
            urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/" + name, p)
        except Exception as e:
            print(f"  ⚠️ tải lỗi {name}: {e}"); return None
    return p if os.path.exists(p) else None

print("📥 Đang tải video...")
GROCERY  = _dl("grocery-store.mp4")
MARKET   = _dl("market-square.mp4")
SUBWAY   = _dl("subway.mp4")
PEOPLE_BW= _dl("people-walking.mp4")
VEH1     = _dl("vehicles.mp4")
VEH2     = _dl("vehicles-2.mp4")
TOM      = os.path.join(VID_DIR, "tomatoes_sorting.mp4")
ROL      = os.path.join(VID_DIR, "packages_rollers.mp4")
BELT     = os.path.join(VID_DIR, "packages_belt.mp4")

RESOLUTION = (1280, 720)
MAX_FRAMES = 120

VIDEOS = [
    dict(name="XE_1", path=VEH1, class_names=["car", "truck", "bus", "motorcycle"],
         line=((0, 50), (100, 50)),
         zone=[(20, 30), (80, 30), (80, 80), (20, 80)]),
    dict(name="XE_2", path=VEH2, class_names=["car", "truck", "bus", "motorcycle"],
         line=((3.5, 93), (93, 89)),
         zone=[(10, 60), (90, 60), (90, 95), (10, 95)]),
    dict(name="NGUOI_MARKET", path=MARKET, class_names=["person"],
         line=((0, 55), (100, 55)),
         zone=[(25, 30), (75, 30), (75, 85), (25, 85)]),
    dict(name="NGUOI_SUBWAY", path=SUBWAY, class_names=["person"],
         line=((0, 50), (100, 50)),
         zone=[(20, 25), (80, 25), (80, 80), (20, 80)]),
    dict(name="NGUOI_BW", path=PEOPLE_BW, class_names=["person"],
         line=((0, 50), (100, 50)),
         zone=[(15, 20), (85, 20), (85, 80), (15, 80)]),
    dict(name="HANG_KE", path=GROCERY, class_names=["bottle", "cup", "bowl", "vase"],
         line=((50, 0), (50, 100)),
         zone=[(20, 25), (80, 25), (80, 75), (20, 75)]),
    dict(name="CA_CHUA", path=TOM, class_names=None,
         line=((0, 70), (100, 70)),
         zone=[(10, 40), (90, 40), (90, 85), (10, 85)]),
    dict(name="KIEN_HANG_1", path=ROL, class_names=None,
         line=((50, 0), (50, 100)),
         zone=[(20, 20), (80, 20), (80, 80), (20, 80)]),
    dict(name="KIEN_HANG_2", path=BELT, class_names=None,
         line=((0, 60), (100, 60)),
         zone=[(15, 30), (85, 30), (85, 80), (15, 80)]),
]

print("\n📋 Danh sách video:")
for v in VIDEOS:
    exists = v["path"] is not None and os.path.exists(v["path"])
    status = "✅" if exists else "❌ THIẾU"
    cls = ", ".join(v["class_names"]) if v["class_names"] else "(tất cả class)"
    print(f"  {status} {v['name']:18} | classes: {cls}")

In [ ]:
# Cell 6 — Tracker, Smoother, Annotator helpers
def make_tracker():
    for kw in (
        dict(track_activation_threshold=0.1, minimum_consecutive_frames=1, lost_track_buffer=120),
        dict(track_thresh=0.1),
        {},
    ):
        try:
            return sv.ByteTrack(**kw)
        except TypeError:
            continue
    return sv.ByteTrack()

def make_smoother():
    try:
        return sv.DetectionsSmoother(length=6)
    except Exception:
        return None

def make_annotators():
    return {
        "box": sv.RoundBoxAnnotator(color_lookup=sv.ColorLookup.TRACK, thickness=2),
        "label": sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_scale=0.5),
        "trace": sv.TraceAnnotator(color_lookup=sv.ColorLookup.TRACK, thickness=2, trace_length=30),
    }

print("✅ Helpers sẵn sàng.")

In [ ]:
# Cell 7 — Hàm đếm: LINE (vạch)
def count_line(video_cfg, max_frames=MAX_FRAMES):
    path = video_cfg["path"]; name = video_cfg["name"]; cls = video_cfg["class_names"]
    (sx_pct, sy_pct), (ex_pct, ey_pct) = video_cfg["line"]
    w, h = RESOLUTION
    sx, sy = int(sx_pct / 100 * w), int(sy_pct / 100 * h)
    ex, ey = int(ex_pct / 100 * w), int(ey_pct / 100 * h)

    tracker = make_tracker(); smoother = make_smoother()
    line = sv.LineZone(start=sv.Point(sx, sy), end=sv.Point(ex, ey))
    annos = make_annotators()
    try: line_anno = sv.LineZoneAnnotator(thickness=2, text_scale=0.8)
    except: line_anno = None

    out_mp4 = os.path.join(OUTDIR, f"{name}_line.mp4")
    cap = cv2.VideoCapture(path)
    writer = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))
    seen = set(); total_det = 0; frame_i = 0

    while frame_i < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, (w, h))
        dets = yolo_detect(fr, cls); total_det += len(dets)
        dets = tracker.update_with_detections(dets)
        if smoother:
            try: dets = smoother.update_with_detections(dets)
            except: pass
        if dets.tracker_id is not None:
            for t in dets.tracker_id:
                if t is not None: seen.add(int(t))
        line.trigger(dets)
        out = fr.copy()
        if len(dets):
            out = annos["trace"].annotate(out, dets)
            out = annos["box"].annotate(out, dets)
            labels = [f"{get_class_labels(dets)[i]} #{int(dets.tracker_id[i])}" if dets.tracker_id is not None else get_class_labels(dets)[i] for i in range(len(dets))]
            out = annos["label"].annotate(out, dets, labels=labels)
        if line_anno: out = line_anno.annotate(out, line)
        else: cv2.line(out, (sx, sy), (ex, ey), (0, 255, 255), 3)
        cv2.rectangle(out, (0, 0), (w, 30), (0, 0, 0), -1)
        cv2.putText(out, f"LINE | IN:{line.in_count} OUT:{line.out_count} total:{line.in_count+line.out_count} tracks:{len(seen)} f:{frame_i+1}",
                    (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        writer.write(out); frame_i += 1
    cap.release(); writer.release()
    return dict(mode="LINE", name=name, in_count=int(line.in_count), out_count=int(line.out_count),
                total=int(line.in_count + line.out_count), tracks=len(seen),
                det_per_frame=round(total_det / max(frame_i, 1), 1), frames=frame_i, mp4=out_mp4)

print("✅ count_line sẵn sàng.")

In [ ]:
# Cell 8 — Hàm đếm: ZONE (vùng)
def count_zone(video_cfg, max_frames=MAX_FRAMES):
    path = video_cfg["path"]; name = video_cfg["name"]; cls = video_cfg["class_names"]
    zone_pct = video_cfg["zone"]; w, h = RESOLUTION
    polygon = np.array([(int(x / 100 * w), int(y / 100 * h)) for x, y in zone_pct], dtype=np.int32)

    tracker = make_tracker(); smoother = make_smoother()
    try: pzone = sv.PolygonZone(polygon=polygon, triggering_anchors=[sv.Position.BOTTOM_CENTER])
    except TypeError:
        try: pzone = sv.PolygonZone(polygon=polygon, frame_resolution_wh=(w, h))
        except TypeError: pzone = sv.PolygonZone(polygon=polygon)

    annos = make_annotators()
    try: zone_anno = sv.PolygonZoneAnnotator(zone=pzone, color=sv.Color.GREEN, thickness=2)
    except: zone_anno = None

    out_mp4 = os.path.join(OUTDIR, f"{name}_zone.mp4")
    cap = cv2.VideoCapture(path)
    writer = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))
    seen = set(); peak = 0; current = 0; total_det = 0; frame_i = 0

    while frame_i < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, (w, h))
        dets = yolo_detect(fr, cls); total_det += len(dets)
        dets = tracker.update_with_detections(dets)
        if smoother:
            try: dets = smoother.update_with_detections(dets)
            except: pass
        if dets.tracker_id is not None:
            for t in dets.tracker_id:
                if t is not None: seen.add(int(t))
        is_in = pzone.trigger(dets)
        current = int(np.sum(is_in)) if is_in is not None else 0
        peak = max(peak, current)
        out = fr.copy()
        if zone_anno: out = zone_anno.annotate(scene=out)
        else:
            cv2.polylines(out, [polygon], True, (0, 255, 0), 2)
            overlay = out.copy(); cv2.fillPoly(overlay, [polygon], (0, 170, 0))
            cv2.addWeighted(overlay, 0.2, out, 0.8, 0, out)
        if len(dets):
            out = annos["trace"].annotate(out, dets)
            out = annos["box"].annotate(out, dets)
            labels = [f"{get_class_labels(dets)[i]} #{int(dets.tracker_id[i])}" if dets.tracker_id is not None else get_class_labels(dets)[i] for i in range(len(dets))]
            out = annos["label"].annotate(out, dets, labels=labels)
        cv2.rectangle(out, (0, 0), (w, 30), (0, 0, 0), -1)
        cv2.putText(out, f"ZONE | in_zone:{current} peak:{peak} tracks:{len(seen)} f:{frame_i+1}",
                    (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        writer.write(out); frame_i += 1
    cap.release(); writer.release()
    return dict(mode="ZONE", name=name, zone_current=current, zone_peak=peak,
                tracks=len(seen), det_per_frame=round(total_det / max(frame_i, 1), 1),
                frames=frame_i, mp4=out_mp4)

print("✅ count_zone sẵn sàng.")

In [ ]:
# Cell 9 — Hàm đếm: FULL-SCREEN (toàn màn hình)
def count_fullscreen(video_cfg, max_frames=MAX_FRAMES):
    path = video_cfg["path"]; name = video_cfg["name"]; cls = video_cfg["class_names"]
    w, h = RESOLUTION
    tracker = make_tracker(); smoother = make_smoother(); annos = make_annotators()

    out_mp4 = os.path.join(OUTDIR, f"{name}_full.mp4")
    cap = cv2.VideoCapture(path)
    writer = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))
    seen = set(); total_det = 0; frame_i = 0; max_concurrent = 0

    while frame_i < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, (w, h))
        dets = yolo_detect(fr, cls); total_det += len(dets)
        dets = tracker.update_with_detections(dets)
        if smoother:
            try: dets = smoother.update_with_detections(dets)
            except: pass
        current_count = len(dets)
        max_concurrent = max(max_concurrent, current_count)
        if dets.tracker_id is not None:
            for t in dets.tracker_id:
                if t is not None: seen.add(int(t))
        out = fr.copy()
        if len(dets):
            out = annos["trace"].annotate(out, dets)
            out = annos["box"].annotate(out, dets)
            labels = [f"{get_class_labels(dets)[i]} #{int(dets.tracker_id[i])}" if dets.tracker_id is not None else get_class_labels(dets)[i] for i in range(len(dets))]
            out = annos["label"].annotate(out, dets, labels=labels)
        cv2.rectangle(out, (0, 0), (w, 30), (0, 0, 0), -1)
        cv2.putText(out, f"FULL | now:{current_count} max:{max_concurrent} unique:{len(seen)} f:{frame_i+1}",
                    (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        writer.write(out); frame_i += 1
    cap.release(); writer.release()
    return dict(mode="FULL", name=name, max_concurrent=max_concurrent, unique_tracks=len(seen),
                tracks=len(seen), det_per_frame=round(total_det / max(frame_i, 1), 1),
                frames=frame_i, mp4=out_mp4)

print("✅ count_fullscreen sẵn sàng.")

In [ ]:
# Cell 10 — Hàm nhận diện lưới (ảnh tĩnh)
N_GRID_FRAMES = 4

def detect_grid(video_cfg, n_frames=N_GRID_FRAMES):
    path = video_cfg["path"]; cls = video_cfg["class_names"]; w, h = RESOLUTION
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    frames = []
    for f_idx in np.linspace(total * 0.2, total * 0.8, n_frames).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f_idx)); ok, fr = cap.read()
        if ok: frames.append(cv2.resize(fr, (w, h)))
    cap.release()

    total_det = 0; best_count = -1
    best_img = frames[0].copy() if frames else np.zeros((h, w, 3), dtype=np.uint8)
    for fr in frames:
        dets = yolo_detect(fr, cls); total_det += len(dets)
        if len(dets) > best_count:
            best_count = len(dets); img = fr.copy()
            for i in range(len(dets)):
                x1, y1, x2, y2 = dets.xyxy[i].astype(int)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, get_class_labels(dets)[i], (x1, max(12, y1 - 4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
            best_img = img
    dpf = round(total_det / max(len(frames), 1), 1)
    cls_str = ", ".join(cls) if cls else "(all)"
    status = "✅" if dpf > 0 else "❌"
    return (cls_str, dpf, status), best_img

print("✅ detect_grid sẵn sàng.")

In [ ]:
# Cell 11 — CHẠY tất cả: 9 video × 3 chế độ + ảnh nhận diện
OUTDIR = os.path.join(WORK, "hardq_out"); os.makedirs(OUTDIR, exist_ok=True)
all_results = []

for v in VIDEOS:
    name = v["name"]; path = v["path"]
    print(f"\n{'='*70}")
    print(f"▶ {name} — {os.path.basename(str(path))}")
    if path is None or not os.path.exists(path):
        print("   ❌ thiếu video, bỏ qua"); continue

    info, grid_img = detect_grid(v)
    grid_png = os.path.join(OUTDIR, f"{name}_detect.png")
    cv2.imwrite(grid_png, grid_img)
    print(f"   Nhận diện: {info[2]} classes={info[0]} det/frame={info[1]}")

    print(f"   Đếm LINE ({MAX_FRAMES} frames)...")
    r1 = count_line(v); r1["grid_png"] = grid_png
    print(f"   → LINE: IN={r1['in_count']} OUT={r1['out_count']} total={r1['total']} tracks={r1['tracks']}")
    all_results.append(r1)

    print(f"   Đếm ZONE ({MAX_FRAMES} frames)...")
    r2 = count_zone(v); r2["grid_png"] = grid_png
    print(f"   → ZONE: in_zone={r2['zone_current']} peak={r2['zone_peak']} tracks={r2['tracks']}")
    all_results.append(r2)

    print(f"   Đếm FULL ({MAX_FRAMES} frames)...")
    r3 = count_fullscreen(v); r3["grid_png"] = grid_png
    print(f"   → FULL: max={r3['max_concurrent']} unique={r3['unique_tracks']}")
    all_results.append(r3)

print(f"\n{'='*70}\n✅ Hoàn tất {len(all_results)} test cases!")

In [ ]:
# Cell 12 — HIỂN THỊ kết quả: ẢNH + VIDEO
for r in all_results:
    name = r["name"]; mode = r["mode"]
    if mode == "LINE":
        desc = f"IN={r['in_count']} OUT={r['out_count']} total={r['total']}"
    elif mode == "ZONE":
        desc = f"in_zone={r['zone_current']} peak={r['zone_peak']}"
    else:
        desc = f"max={r['max_concurrent']} unique={r['unique_tracks']}"
    display(Markdown(f"### {name} · {mode} — {desc} · tracks={r['tracks']}"))
    grid = r.get("grid_png")
    if grid and os.path.exists(grid):
        display(IPImage(filename=grid, width=700))
    mp4 = r["mp4"]; h264 = mp4.replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", mp4,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264], check=False)
    show = h264 if os.path.exists(h264) else mp4
    display(Video(show, embed=True, width=640))
print("\n💾 Tất cả ảnh + video đã lưu ở:", OUTDIR)

In [ ]:
# Cell 13 — BẢNG TỔNG HỢP
print("\n" + "=" * 90)
print("BẢNG TỔNG HỢP — 9 VIDEO × 3 CHẾ ĐỘ ĐẾM (YOLO + supervision)")
print("=" * 90)
print(f"{'Video':18} {'Mode':6} {'Kết quả':30} {'Tracks':>7} {'Det/fr':>7} {'Frames':>7}")
print("-" * 90)
for r in all_results:
    name = r["name"]; mode = r["mode"]
    if mode == "LINE":
        result = f"IN={r['in_count']} OUT={r['out_count']} total={r['total']}"
    elif mode == "ZONE":
        result = f"in_zone={r['zone_current']} peak={r['zone_peak']}"
    else:
        result = f"max={r['max_concurrent']} unique={r['unique_tracks']}"
    tracks = r["tracks"]
    print(f"{name:18} {mode:6} {result:30} {tracks:>7} {r['det_per_frame']:>7} {r['frames']:>7}")
n_video = len(set(r["name"] for r in all_results))
print(f"\n📊 Tổng: {n_video} video × 3 chế độ = {len(all_results)} test cases")
print("📝 Lưu ý: Video sản phẩm (cà chua, kiện hàng) dùng detect tất cả class YOLO thấy.")
print("   Nếu det/frame ≈ 0 → cần LocateAnything (open-vocab) cho loại vật thể này.")

### Đọc kết quả

- **LINE**: Đếm IN/OUT = số object cắt qua vạch theo 2 hướng. `total` = IN + OUT.
- **ZONE**: `in_zone` = số object trong vùng ở frame cuối, `peak` = số cao nhất từng đạt.
- **FULL**: `max_concurrent` = số object đồng thời nhiều nhất, `unique` = tổng object khác nhau.
- **tracks**: Tổng số danh tính ByteTrack đã gán.
- **det/frame ≈ 0**: YOLO không bắt được loại vật này → cần LocateAnything.
- **Đếm sai nhiều**: Thử đổi `CONF_THRESHOLD` (Cell 4) hoặc dùng `yolov8l.pt` (Cell 3).
- **OOM**: Hạ `RESOLUTION` xuống `(960, 540)` ở Cell 5.